In [ ]:
# Install dependencies
%pip install snowflake-connector-python snowflake-sqlalchemy==1.5.1

# NaverCafe NLP Pipeline
Crawls community posts mentioning diaper brands, classifies by brand, detects leakage/risk signals, and uploads to data warehouse.

**Pipeline steps:**
1. Fetch recent community posts from data warehouse
2. Brand detection via keyword matching
3. Feature engineering: leakage flag, media flag, risk flag, risk keyword extraction
4. Deduplication against existing records
5. Upload to data warehouse

In [ ]:
import os
import re
import pandas as pd
import datetime
import snowflake.connector
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine
from snowflake.connector.pandas_tools import pd_writer

# Current time in KST (UTC+9)
now = datetime.datetime.now() + datetime.timedelta(hours=9)
five_days_ago_str = (now - datetime.timedelta(days=5)).strftime('%Y-%m-%d')

In [ ]:
# ── DATABASE CONNECTION ──────────────────────────────────────────────────
# Set environment variables before running:
#   SNOWFLAKE_USER, SNOWFLAKE_PASSWORD, SNOWFLAKE_ACCOUNT
#   SNOWFLAKE_DATABASE, SNOWFLAKE_SCHEMA, SNOWFLAKE_WAREHOUSE

con = snowflake.connector.connect(
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE']
)
cur = con.cursor()

In [ ]:
# ── DATA EXTRACTION ──────────────────────────────────────────────────────
# Fetch community posts from the past 5 days
# Excludes deal boards, promotional posts, and review-heavy threads

content_query = f"""
SELECT SEQ, POST_NUMBER, BOARD_PATH, CAFE_NAME, TITLE, CONTENTS,
       MEDIA, AUTHOR, URL, WDATE, RDATE, VIEWS, COMMENT_CNT, LIKES
FROM COMMUNITY_POSTS_VIEW
WHERE RDATE >= '{five_days_ago_str}'
AND NOT (
   BOARD_PATH IN ('Deal Board')
   OR BOARD_PATH LIKE '%HotDeal%'
   OR TITLE LIKE '%postpartum care%'
   OR TITLE LIKE '%baby products%'
   OR TITLE LIKE '%review%'
   OR TITLE LIKE '%hot deal%'
);
"""

content = pd.DataFrame(cur.execute(content_query))
content_columns = ["SEQ", "POST_NUMBER", "BOARD_PATH", "CAFE_NAME", "TITLE",
                   "CONTENTS", "MEDIA", "AUTHOR", "URL", "WDATE", "RDATE",
                   "VIEWS", "COMMENT_CNT", "LIKES"]
content.columns = content_columns
content['CONTENTS'] = content['CONTENTS'].fillna('')
content['RDATE'] = pd.to_datetime(content['RDATE'], format='%Y-%m-%d').dt.strftime('%Y-%m-%d')
content['WDATE'] = pd.to_datetime(content['WDATE'], format='%Y-%m-%d').dt.strftime('%Y-%m-%d')
content['YEAR'] = pd.to_datetime(content['WDATE']).dt.year
content['MONTH'] = pd.to_datetime(content['WDATE']).dt.month

In [ ]:
# ── BRAND KEYWORD DICTIONARY ─────────────────────────────────────────────
# Each brand maps to a regex pattern of known aliases and product line names
# Korean brand names are retained as the source data is in Korean

BRAND_KEYWORDS = {
    'Huggies':  '|'.join(['하기스','huggies','맥스드라이','매직드라이','네이처메이드','naturemade','매직컴포트','magiccomfort']),
    'Pampers':  '|'.join(['펨퍼스','팸퍼스','아르모니','베이비드라이','터치오브네이처','에어차차','통잠팬티']),
    'Penelope': '|'.join(['페넬로페','미라클올데이','미라클 올데이','씬씬씬']),
    'Mamipoko': '|'.join(['마미포코','땀먹는팬티','에어핏공기솔솔','리프가닉']),
    'Bosomi':   '|'.join(['보솜이','원더바이원더','메가드라이','리얼코튼오가니크']),
    'Kindoh':   '|'.join(['킨도','업앤플레이','업 앤 플레이'])
}

TARGET_BRANDS = list(BRAND_KEYWORDS.keys())

def detect_brands(content_text, title):
    """Return list of brands mentioned in the content or title."""
    detected = []
    for brand, pattern in BRAND_KEYWORDS.items():
        if re.search(pattern, content_text, re.IGNORECASE) or re.search(pattern, title, re.IGNORECASE):
            detected.append(brand)
    return detected

# Expand rows: one row per brand mention per post
expanded_rows = []
for _, row in content.iterrows():
    for brand in detect_brands(row['CONTENTS'], row['TITLE']):
        expanded_row = row.copy()
        expanded_row['BRAND'] = brand
        expanded_rows.append(expanded_row)

expanded_content = pd.DataFrame(expanded_rows)

# Create unique ID per post-brand combination
expanded_content['UNIQUE_ID'] = expanded_content.apply(
    lambda row: f"{row['SEQ']}-{row['BRAND']}", axis=1
)

# Filter to target brands only
expanded_content = expanded_content[expanded_content['BRAND'].isin(TARGET_BRANDS)]

In [ ]:
# ── FEATURE ENGINEERING ─────────────────────────────────────────────────
# Leakage flag: detects posts mentioning diaper leakage incidents
# Risk flag: detects posts mentioning foreign objects or contamination

LEAKAGE_KEYWORDS = '|'.join([
    '새어','새는','샜다','샌적','샐까요','새더라','줄줄','흘러','넘치고',
    '젖어','젖었','젖네','이불','역류','흥건','축축'
])

RISK_KEYWORDS = ['이물질','녹물','본드','곰팡이','벌레','날파리','쇳가루']

def check_leakage(row):
    """Return 1 if post mentions leakage, else 0."""
    return int(bool(
        re.search(LEAKAGE_KEYWORDS, row['CONTENTS'], re.IGNORECASE) or
        re.search(LEAKAGE_KEYWORDS, row['TITLE'], re.IGNORECASE)
    ))

def find_risk_keywords(content_text, title):
    """Return comma-separated string of detected risk keywords."""
    found = {kw for kw in RISK_KEYWORDS
             if re.search(kw, content_text, re.IGNORECASE) or re.search(kw, title, re.IGNORECASE)}
    return ', '.join(found)

def find_nearby_sentences(text, keywords, window=2):
    """Extract sentences surrounding a keyword match (context window)."""
    sentences = re.split(r'(?<=[.!?]) +', text)
    keyword_indices = [
        i for i, s in enumerate(sentences)
        if any(re.search(kw, s, re.IGNORECASE) for kw in keywords)
    ]
    nearby = set()
    for idx in keyword_indices:
        for i in range(max(0, idx - window), min(len(sentences), idx + window + 1)):
            nearby.add(sentences[i])
    return ' '.join(nearby)

def check_risk(row):
    """Return 1 if a risk keyword appears near a brand mention, else 0."""
    nearby_text = ''
    for brand, pattern in BRAND_KEYWORDS.items():
        if re.search(pattern, row['CONTENTS'], re.IGNORECASE):
            nearby_text += ' ' + find_nearby_sentences(row['CONTENTS'], [pattern])
        if re.search(pattern, row['TITLE'], re.IGNORECASE):
            nearby_text += ' ' + find_nearby_sentences(row['TITLE'], [pattern])
    return int(any(re.search(rk, nearby_text, re.IGNORECASE) for rk in RISK_KEYWORDS))

# Brand keyword dict split by brand for sentence extraction
brand_keywords_list = {brand: pattern.split('|') for brand, pattern in BRAND_KEYWORDS.items()}

def extract_brand_sentence(row):
    """Extract the sentence(s) most relevant to the detected brand."""
    keywords_for_brand = brand_keywords_list.get(row['BRAND'], [])
    sentences = re.split(r'[;.!?\n]', row['CONTENTS'])
    for i, s in enumerate(sentences):
        if any(kw in s for kw in keywords_for_brand):
            prev_s = sentences[i-1] if i > 0 else ''
            next_s = sentences[i+1] if i < len(sentences)-1 else ''
            return ' '.join(filter(None, [prev_s, s, next_s])).strip()
    return row['TITLE']

expanded_content['LEAKAGE_YN'] = expanded_content.apply(check_leakage, axis=1)
expanded_content['MEDIA_YN'] = expanded_content['MEDIA'].apply(
    lambda x: 1 if pd.notnull(x) and x != '' else 0
)
expanded_content['RISK_YN'] = expanded_content.apply(check_risk, axis=1)
expanded_content['RISK_KEYWORD'] = expanded_content.apply(
    lambda row: find_risk_keywords(row['CONTENTS'], row['TITLE']), axis=1
)
expanded_content['CONTENTS_S'] = expanded_content.apply(extract_brand_sentence, axis=1)

In [ ]:
# ── DEDUPLICATION ────────────────────────────────────────────────────────
# Load existing records from the warehouse and merge,
# keeping the latest version of each post-brand pair

orig_query = "SELECT * FROM NLP_TABLE;"
orig_columns = ["UNIQUE_ID", "SEQ", "POST_NUMBER", "BOARD_PATH", "CAFE_NAME",
                "TITLE", "CONTENTS", "MEDIA", "AUTHOR", "URL", "WDATE", "RDATE",
                "VIEWS", "COMMENT_CNT", "LIKES", "YEAR", "MONTH", "BRAND",
                "LEAKAGE_YN", "MEDIA_YN", "RISK_YN", "RISK_KEYWORD", "CONTENTS_S"]

orig_content = pd.DataFrame(cur.execute(orig_query).fetchall(), columns=orig_columns)

combined = pd.concat(
    [orig_content.set_index(['POST_NUMBER', 'TITLE', 'CONTENTS', 'BRAND']),
     expanded_content.set_index(['POST_NUMBER', 'TITLE', 'CONTENTS', 'BRAND'])],
    axis=0
).reset_index()

updated_content = combined.drop_duplicates(
    subset=['POST_NUMBER', 'TITLE', 'CONTENTS', 'BRAND'], keep='last'
)

In [ ]:
# ── UPLOAD TO DATA WAREHOUSE ─────────────────────────────────────────────
# Truncate the existing table and replace with updated records

engine = create_engine(URL(
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
))

table_name = 'NLP_TABLE'

with engine.connect() as con:
    con.execute(f'DELETE FROM {table_name}')
    updated_content.to_sql(name=table_name, con=con, if_exists='append',
                           method=pd_writer, index=False)

print(f'Uploaded {len(updated_content)} records to {table_name}')